# Running inference on the Kria KV260 DPU 

The input images are grayscale (the same preprocessing as in training).

## Import libraries

Run this notebook with the PYNQ kernel *Python 3 (ipykernel)*. The board has no direct internet access; pip, apt and git use the UT proxy (already configured).

In [ ]:
import os
import pickle
from time import time

import numpy as np
import matplotlib.pyplot as plt


## Define the preprocessing and the CIFAR-10 loader
The preprocessing must be **identical in every step** (training, quantization, CPU and DPU inference): same `rgb_to_gray` function, then division by 255.

In [ ]:
def rgb_to_gray(x):
    """uint8 RGB images (..., 32, 32, 3) -> uint8 grayscale (..., 32, 32, 1), integer arithmetic (same in all steps)."""
    x = x.astype(np.uint32)
    gray = (x[..., 0] * 299 + x[..., 1] * 587 + x[..., 2] * 114 + 500) // 1000
    return gray.astype(np.uint8)[..., np.newaxis]


def load_cifar10_test(directory):
    """Test batch of the CIFAR-10 python version (cifar-10-python.tar.gz -> cifar-10-batches-py/)."""
    with open(os.path.join(directory, "test_batch"), "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    x = batch[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)   # (N, 32, 32, 3) uint8 RGB
    return x, np.array(batch[b"labels"])


## Load CIFAR-10 Dataset

In [ ]:
# Folder with the extracted dataset (tar -xzf cifar-10-python.tar.gz), relative to this notebook
directory = 'cifar-10-batches-py'

x_test_rgb, y_test = load_cifar10_test(directory)
x_test = rgb_to_gray(x_test_rgb)
print(f"Test samples: {x_test.shape[0]}, shape {x_test.shape[1:]}")


## Normalize Image Data

In [ ]:
# Normalize images to range [0,1] (same as in training)
x_test_norm = (x_test / 255.0).astype("float32")
print(x_test_norm.shape, "normalization complete.")


## Load DPU Overlay and Model

The DPU overlay is the bitstream that turns the FPGA into a deep-learning processing unit. `DpuOverlay("dpu.bit")` loads the default DPU of pynq-dpu (DPUCZDX8G, B4096), which matches the default KV260 `arch.json`. If you built your own DPU (step 3b), copy its `dpu.bit`, `dpu.hwh` and `dpu.xclbin`, not renamed, into a subfolder such as `my_dpu/` and set `BITSTREAM = "my_dpu/dpu.bit"`. The `.xmodel` must be compiled for the DPU you load.

In [ ]:
from pynq_dpu import DpuOverlay

BITSTREAM = "dpu.bit"                       # default DPU of pynq-dpu, or "my_dpu/dpu.bit" (your own DPU)
XMODEL = "cifar10_bw_model_compiled.xmodel"  # your compiled model (output of vai_c_tensorflow2)

overlay = DpuOverlay(BITSTREAM)
overlay.load_model(XMODEL)
dpu = overlay.runner
print("DPU model loaded successfully.")


## Get DPU Input/Output Tensor Details

In [ ]:
# Get tensor information
inputTensors = dpu.get_input_tensors()
outputTensors = dpu.get_output_tensors()

shapeIn = tuple(inputTensors[0].dims)
shapeOut = tuple(outputTensors[0].dims)
outputSize = int(outputTensors[0].get_data_size() / shapeIn[0])

softmax = np.empty(outputSize)

# Prepare memory buffers
output_data = [np.empty(shapeOut, dtype=np.float32, order="C")]
input_data = [np.empty(shapeIn, dtype=np.float32, order="C")]

print(f"Input shape: {shapeIn}, Output shape: {shapeOut}")


## Define Softmax Function

In [ ]:
def getSoftmax(x):
    exp_x = np.exp(x - np.max(x))  # for numerical stability
    return exp_x / exp_x.sum()


## Run Inference on a Few Sample Images

In [ ]:
%matplotlib inline

# CIFAR-10 class names
class_names = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

Num_images = 10  # Number of samples to visualize
image = np.zeros((1, 32, 32, 1), dtype=np.float32)
fig, ax = plt.subplots(1, Num_images, figsize=(15, 5))
plt.tight_layout()

print("Started inference on sample images...\n")

for i in range(Num_images):
    image[0, ...] = x_test_norm[i]
    input_data = [image]
    
    # Run inference
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)

    # Apply softmax
    temp = [j.reshape(1, outputSize) for j in output_data]
    softmax = getSoftmax(temp[0][0])
    pred = softmax.argmax()

    # Get predicted class name
    class_name = class_names[pred]

    # Display image with predicted class
    ax[i].imshow(x_test[i].squeeze(), cmap='gray')
    ax[i].set_title(f'{class_name}\n({pred})', fontsize=10)
    ax[i].axis('off')

    # Print results to console as well
    print(f"Image {i+1}: Predicted class = {class_name} (index {pred})")

print("\nCompleted inference.")
plt.show()



## Full Test Set Inference and Performance Metrics

In [ ]:
N = x_test_norm.shape[0]
y_pred = np.empty_like(y_test)
image = np.zeros((1, 32, 32, 1), dtype=np.float32)
print("Started full DPU inference...")
start = time()


for i in range(N):
    image[0, ...] = x_test_norm[i]
    input_data = [image]
    
    job_id = dpu.execute_async(input_data, output_data)
    dpu.wait(job_id)
    
    temp = [j.reshape(1, outputSize) for j in output_data]
    softmax = getSoftmax(temp[0][0])
    y_pred[i] = softmax.argmax()

stop = time()
print("Inference complete.")

## Evaluate Accuracy and Throughput

In [ ]:
correct = np.sum(y_pred == y_test)
acc = (correct / N) * 100
exec_time = stop - start
fps = N / exec_time

print(f"Accuracy: {acc:.2f}%")
print(f"Execution time: {exec_time:.2f} seconds")
print(f"Average inference time per image: {1000.0 * exec_time / N:.2f} ms")
print(f"Throughput: {fps:.2f} FPS")

## Clean Up Resources

In [ ]:
del overlay
del dpu
print("Released DPU resources.")